# RMSSD Validation: PPG vs ECG

This notebook compares RMSSD calculated from our PPG sensor against an ECG sensor (BrainFlow) recorded from the same person at the same time.

**Workflow:**
1. Load PPG data from `ppg_data.txt` and ECG data from an Excel file
2. Detect tap artifacts in both signals to synchronize them
3. Trim both signals to the same duration after the tap
4. Process each signal with NeuroKit2 to detect peaks
5. Calculate RMSSD from each signal
6. Compare with absolute error

In [ ]:
import neurokit2 as nk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# CONFIGURATION — Change these as needed
# ============================================================
PPG_PATH = "ppg_data.txt"                                    # Path to PPG sensor data (one value per line)
ECG_EXCEL_PATH = "BrainFlow-RAW_2025-11-25_export_data_0.xlsx"  # Path to ECG Excel export
PPG_FS = 100   # PPG sampling rate in Hz
ECG_FS = 250   # ECG sampling rate in Hz
RMSSD_DURATION_SEC = 60  # How many seconds of data (after sync) to use for RMSSD calculation

## Load Data

In [ ]:
# --- Load PPG data ---
def load_ppg(path):
    with open(path, "r") as f:
        data = [float(line.strip()) for line in f if line.strip()]
    return np.array(data, dtype=float)

ppg_raw = load_ppg(PPG_PATH)

# --- Load ECG data from Excel ---
df = pd.read_excel(ECG_EXCEL_PATH)
ecg_raw = df.iloc[1:, 1].dropna().to_numpy(dtype=float)

print(f"PPG: {len(ppg_raw)} samples ({len(ppg_raw)/PPG_FS:.1f} seconds at {PPG_FS} Hz)")
print(f"ECG: {len(ecg_raw)} samples ({len(ecg_raw)/ECG_FS:.1f} seconds at {ECG_FS} Hz)")

## Tap Synchronization

Both sensors are tapped simultaneously at the start of recording. This creates a large artifact spike in both signals. We detect the first large spike in each signal's absolute derivative and align them.

In [ ]:
def detect_tap(signal, fs, threshold_multiplier=5):
    """Detect the first large artifact (tap) in a signal using the absolute derivative."""
    deriv = np.abs(np.diff(signal))
    threshold = np.mean(deriv) + threshold_multiplier * np.std(deriv)
    tap_candidates = np.where(deriv > threshold)[0]
    if len(tap_candidates) == 0:
        print("WARNING: No tap detected! Using index 0.")
        return 0
    return tap_candidates[0]

# Detect taps
ppg_tap_idx = detect_tap(ppg_raw, PPG_FS)
ecg_tap_idx = detect_tap(ecg_raw, ECG_FS)

print(f"PPG tap detected at sample {ppg_tap_idx} ({ppg_tap_idx/PPG_FS:.2f} s)")
print(f"ECG tap detected at sample {ecg_tap_idx} ({ecg_tap_idx/ECG_FS:.2f} s)")

# Plot raw signals with tap markers
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

ppg_time = np.arange(len(ppg_raw)) / PPG_FS
axes[0].plot(ppg_time, ppg_raw, color='steelblue')
axes[0].axvline(ppg_tap_idx / PPG_FS, color='red', linestyle='--', linewidth=2, label=f'Tap @ {ppg_tap_idx/PPG_FS:.2f}s')
axes[0].set_title('Raw PPG Signal with Tap Detection')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')
axes[0].legend()

ecg_time = np.arange(len(ecg_raw)) / ECG_FS
axes[1].plot(ecg_time, ecg_raw, color='darkorange')
axes[1].axvline(ecg_tap_idx / ECG_FS, color='red', linestyle='--', linewidth=2, label=f'Tap @ {ecg_tap_idx/ECG_FS:.2f}s')
axes[1].set_title('Raw ECG Signal with Tap Detection')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Amplitude')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Trim signals: start from tap, take RMSSD_DURATION_SEC seconds
ppg_start = ppg_tap_idx
ppg_end = ppg_start + RMSSD_DURATION_SEC * PPG_FS
ppg_trimmed = ppg_raw[ppg_start:ppg_end]

ecg_start = ecg_tap_idx
ecg_end = ecg_start + RMSSD_DURATION_SEC * ECG_FS
ecg_trimmed = ecg_raw[ecg_start:ecg_end]

print(f"PPG trimmed: {len(ppg_trimmed)} samples ({len(ppg_trimmed)/PPG_FS:.1f} s)")
print(f"ECG trimmed: {len(ecg_trimmed)} samples ({len(ecg_trimmed)/ECG_FS:.1f} s)")

## Process PPG Signal

In [ ]:
# Process PPG with NeuroKit2
signals_ppg, info_ppg = nk.ppg_process(ppg_trimmed, sampling_rate=PPG_FS)

# Extract peak indices
ppg_peak_indices = np.where(signals_ppg["PPG_Peaks"] == 1)[0]
print(f"PPG peaks detected: {len(ppg_peak_indices)}")

# Plot PPG with detected peaks
ppg_time_trimmed = np.arange(len(ppg_trimmed)) / PPG_FS

plt.figure(figsize=(16, 5))
plt.plot(ppg_time_trimmed, signals_ppg["PPG_Clean"], color='steelblue', label='Cleaned PPG')
plt.scatter(ppg_time_trimmed[ppg_peak_indices], signals_ppg["PPG_Clean"].iloc[ppg_peak_indices],
            color='red', s=40, zorder=5, label='Detected Peaks')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.title('PPG Signal with Detected Peaks (after sync)')
plt.legend()
plt.tight_layout()
plt.show()

## Process ECG Signal

In [ ]:
# Process ECG with NeuroKit2
signals_ecg, info_ecg = nk.ecg_process(ecg_trimmed, sampling_rate=ECG_FS)

# Extract R-peak indices
ecg_rpeaks = info_ecg["ECG_R_Peaks"]
print(f"ECG R-peaks detected: {len(ecg_rpeaks)}")

# Plot ECG with detected R-peaks
ecg_time_trimmed = np.arange(len(ecg_trimmed)) / ECG_FS
ecg_clean = signals_ecg["ECG_Clean"]

plt.figure(figsize=(16, 5))
plt.plot(ecg_time_trimmed, ecg_clean, color='darkorange', label='Cleaned ECG')
plt.scatter(ecg_time_trimmed[ecg_rpeaks], ecg_clean.iloc[ecg_rpeaks],
            color='red', s=40, zorder=5, label='Detected R-Peaks')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.title('ECG Signal with Detected R-Peaks (after sync)')
plt.legend()
plt.tight_layout()
plt.show()

## Calculate RMSSD & Compare

In [ ]:
# --- PPG RMSSD ---
# Build a peaks dict for nk.hrv_time (expects a dict with "PPG_Peaks" key containing indices)
ppg_peaks_dict = {"PPG_Peaks": ppg_peak_indices}
hrv_ppg = nk.hrv_time(ppg_peaks_dict, sampling_rate=PPG_FS)
rmssd_ppg = hrv_ppg["HRV_RMSSD"].values[0]

# --- ECG RMSSD ---
ecg_peaks_dict = {"ECG_R_Peaks": ecg_rpeaks}
hrv_ecg = nk.hrv_time(ecg_peaks_dict, sampling_rate=ECG_FS)
rmssd_ecg = hrv_ecg["HRV_RMSSD"].values[0]

# --- Comparison ---
absolute_error = abs(rmssd_ppg - rmssd_ecg)
rmse = np.sqrt((rmssd_ppg - rmssd_ecg) ** 2)  # For a single pair, RMSE = absolute error

print("=" * 50)
print(f"  PPG RMSSD:        {rmssd_ppg:.2f} ms")
print(f"  ECG RMSSD:        {rmssd_ecg:.2f} ms")
print(f"  Absolute Error:   {absolute_error:.2f} ms")
print(f"  RMSE:             {rmse:.2f} ms")
print("=" * 50)

# Summary table
results_df = pd.DataFrame({
    "Source": ["PPG", "ECG"],
    "RMSSD (ms)": [rmssd_ppg, rmssd_ecg],
    "Peaks Detected": [len(ppg_peak_indices), len(ecg_rpeaks)],
    "Duration (s)": [len(ppg_trimmed) / PPG_FS, len(ecg_trimmed) / ECG_FS]
})

error_df = pd.DataFrame({
    "Metric": ["Absolute Error (ms)", "RMSE (ms)"],
    "Value": [absolute_error, rmse]
})

print("\n--- Results ---")
display(results_df)
print("\n--- Error ---")
display(error_df)